# Phase3_Profiling

Converted automatically from Python script into notebook format.

In [ ]:
"""
Phase 3 (Cross-platform) — dataset profiling and platform-shift diagnostics.

Mirrors the Phase 2 ten-metric profiling convention from
`P5 Real Life Sets V2/Dataset_Profiling_and_Mapping.ipynb` so the Phase 3
datasets are directly comparable to the Phase 1 L9 anchors and the Phase 2
real-data panel, and additionally computes per-block platform-shift
diagnostics that the within-block training and target platforms cannot share:

    - Median library size (UMI or read count)
    - Sparsity (% zeros) and per-cell gene-detection rate
    - Within-block per-platform kNN purity (separability)
    - Cross-platform gene-overlap Jaccard (relative to the block's
      designated reference platform)

The block structure follows Methods Phase 3:

    CellBench block      (lung cancer cell-line mixtures)
        - 10X v2          [reference]
        - CEL-seq2
        - Drop-seq
    Pancreas block       (cross-cohort, not pure platform contrast)
        - Baron       (inDrop)         [reference]
        - Segerstolpe (Smart-seq2)
    PBMCbench block      (matched PBMC samples)
        - 10X v2          [reference]
        - Drop-seq
        - inDrops
        - Seq-well

Outputs (all in `./` next to this script):
    - Profiling_P9.csv             — ten-metric profile per dataset
    - Phase3_PlatformShift.csv     — block / platform / Δ-vs-reference per dataset
    - Phase3_GeneOverlap.csv       — Jaccard gene-overlap within each block
    - Plots/*.png                  — block-level diagnostic plots

Run:
    python Phase3_Profiling.py
"""

from __future__ import annotations

import gc
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import mmread
from scipy.stats import entropy
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [ ]:
# 1. PATHS, BLOCK STRUCTURE, METRIC SETS

In [ ]:
from pathlib import Path

P9_BASE_DIR = Path("./")      # current working directory
OUT_DIR = P9_BASE_DIR
PLOT_DIR = OUT_DIR / "Plots"

PLOT_DIR.mkdir(parents=True, exist_ok=True)


# Block structure: folder-name -> (block_label, platform_label, is_reference)
# Folder names must match the on-disk subdirectory names in this folder.
BLOCKS = {
    # CellBench
    "Cellbench 10xv2":              ("CellBench", "10X v2",     True),
    "Cellbench Cel-Seq2":           ("CellBench", "CEL-seq2",   False),
    "Cellbench Drop-seq":           ("CellBench", "Drop-seq",   False),
    # Pancreas
    "Baron-Pancreas":               ("Pancreas",  "inDrop",     True),
    "Segerstolpe-Pancreas-2016":    ("Pancreas",  "Smart-seq2", False),
    # PBMCbench
    "PBMCBench 10xv2-8cl":          ("PBMCbench", "10X v2",     True),
    "PBMCBench Drop-seq-8cl":       ("PBMCbench", "Drop-seq",   False),
    "PBMCBench InDrops-8cl":        ("PBMCbench", "inDrops",    False),
    "PBMCBench Seq-well-6cl":       ("PBMCbench", "Seq-well",   False),
    # 6cl 10x is a PBMCbench cell-set variant; profile but treat as auxiliary
    "PBMCBench 10xv2-6cl":          ("PBMCbench", "10X v2 (6cl)", False),
}

# Same ten-metric panel as Phase 2 (Profiling_Real.csv schema).
METRICS_10 = [
    "Total_Cells", "Min_Cluster_Size", "Total_CellTypes",
    "Dropout_Pct", "Median_Library_Per_Cell",
    "Norm_Shannon_Entropy",
    "Silhouette_Score", "KNN_Purity",
    "Mean_Pi_Score", "BCV_Dispersion_Proxy",
]

In [ ]:
# 2. LOADER (mirrors Phase 2)

In [ ]:
def load_real_dataset(dataset_path: Path) -> ad.AnnData:
    counts   = mmread(dataset_path / "counts.mtx").T.tocsr()
    barcodes = pd.read_csv(dataset_path / "barcodes.tsv", header=None, sep="\t")[0].astype(str).values

    gene_file = dataset_path / "genes.tsv"
    if not gene_file.exists():
        gene_file = dataset_path / "features.tsv"
    gene_names = pd.read_csv(gene_file, header=None, sep="\t").iloc[:, 0].astype(str).values

    metadata = pd.read_csv(dataset_path / "metadata.tsv", index_col=0, sep="\t")

    adata = ad.AnnData(X=counts, obs=metadata)
    adata.obs_names = barcodes
    adata.var_names = gene_names

    if "Ground_Truth_Celltype" not in adata.obs.columns:
        for alt in ["cell_type", "celltype", "CellType", "label"]:
            if alt in adata.obs.columns:
                adata.obs["Ground_Truth_Celltype"] = adata.obs[alt].astype(str)
                break
        else:
            raise KeyError(
                f"No recognised cell-type column in {dataset_path}.\n"
                f"Available obs columns: {list(adata.obs.columns)}"
            )

    adata.obs["Ground_Truth_Celltype"] = adata.obs["Ground_Truth_Celltype"].astype(str)
    return adata

In [ ]:
# 3. PROFILE — ten-metric panel + platform-shift extras

In [ ]:
def calc_knn_purity(adata, label_col="Ground_Truth_Celltype", n_neighbors=15):
    assert "X_pca" in adata.obsm, "Run sc.pp.pca() before calc_knn_purity()"
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep="X_pca", metric="euclidean")
    dist   = adata.obsp["distances"]
    labels = adata.obs[label_col].values

    purities = []
    for i in range(adata.n_obs):
        nbr_idx = dist[i, :].nonzero()[1]
        if len(nbr_idx) == 0:
            continue
        purities.append(np.sum(labels[nbr_idx] == labels[i]) / len(nbr_idx))
    return float(np.mean(purities)) if purities else np.nan


def profile_dataset(adata_raw, dataset_name,
                    label_col="Ground_Truth_Celltype",
                    n_hvgs=2000, n_pcs=30, n_neighbors=15):
    result = {"Sample": dataset_name}

    # --- 1. Label-only ---
    cluster_counts = adata_raw.obs[label_col].value_counts()
    n_cells        = adata_raw.n_obs
    n_clusters     = len(cluster_counts)
    props          = cluster_counts.values / n_cells

    result["Total_Cells"]      = n_cells
    result["Total_CellTypes"]  = n_clusters
    result["Min_Cluster_Size"] = int(cluster_counts.min())
    if n_clusters > 1:
        H, H_max = entropy(props, base=np.e), np.log(n_clusters)
        result["Norm_Shannon_Entropy"] = round(H / H_max, 4)
    else:
        result["Norm_Shannon_Entropy"] = np.nan

    # --- 2. Raw-count metrics (sequencing-depth axis) ---
    adata = adata_raw.copy()
    X = adata.X
    if hasattr(X, "toarray"):
        n_zeros   = X.shape[0] * X.shape[1] - X.nnz
        lib_sizes = np.asarray(X.sum(axis=1)).flatten()
        genes_per_cell = np.asarray((X > 0).sum(axis=1)).flatten()
    else:
        n_zeros   = int(np.sum(X == 0))
        lib_sizes = X.sum(axis=1)
        genes_per_cell = (X > 0).sum(axis=1)

    result["Dropout_Pct"]             = round(n_zeros / (adata.n_obs * adata.n_vars) * 100, 2)
    result["Median_Library_Per_Cell"] = round(float(np.median(lib_sizes)), 1)
    # Phase 3 extras (kept inside the row so they live in the same CSV)
    result["Mean_Library_Per_Cell"]   = round(float(np.mean(lib_sizes)), 1)
    result["Median_Genes_Per_Cell"]   = int(np.median(genes_per_cell))
    result["Mean_Genes_Per_Cell"]     = round(float(np.mean(genes_per_cell)), 1)
    result["Total_Genes_Detected"]    = int((np.asarray((X > 0).sum(axis=0)).flatten() > 0).sum())

    # --- 3. Normalise -> HVG selection (seurat flavour) ---
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=n_hvgs, flavor="seurat")
    hvg_mask = adata.var["highly_variable"]
    result["BCV_Dispersion_Proxy"] = round(
        float(np.median(adata.var.loc[hvg_mask, "dispersions_norm"])), 4
    )

    # --- 4. Geometry (scale + PCA on HVG log-norm) ---
    adata_de  = adata[:, hvg_mask].copy()
    adata_geo = adata[:, hvg_mask].copy()
    sc.pp.scale(adata_geo, max_value=10)
    sc.tl.pca(adata_geo, n_comps=min(n_pcs, adata_geo.n_obs - 1, adata_geo.n_vars - 1))
    pca_coords = adata_geo.obsm["X_pca"]
    labels_arr = adata_geo.obs[label_col].values
    if n_clusters > 1:
        result["Silhouette_Score"] = round(silhouette_score(pca_coords, labels_arr), 4)
        result["KNN_Purity"]       = round(calc_knn_purity(adata_geo, label_col, n_neighbors), 4)
    else:
        result["Silhouette_Score"] = np.nan
        result["KNN_Purity"]       = np.nan

    # --- 5. Xiao π score (DE difficulty proxy) ---
    mean_pi = np.nan
    if n_clusters > 1:
        valid_groups = cluster_counts[cluster_counts >= 3].index.tolist()
        if len(valid_groups) > 1:
            sc.tl.rank_genes_groups(adata_de, label_col, groups=valid_groups,
                                    method="wilcoxon", key_added="de")
            pi_scores = []
            for grp in adata_de.uns["de"]["names"].dtype.names:
                pvals = adata_de.uns["de"]["pvals_adj"][grp][:50]
                lfcs  = adata_de.uns["de"]["logfoldchanges"][grp][:50]
                pi_scores.extend(-np.log10(pvals + 1e-300) * np.abs(lfcs))
            if pi_scores:
                mean_pi = round(float(np.mean(pi_scores)), 4)
    result["Mean_Pi_Score"] = mean_pi

    # --- 6. Detected-gene set for cross-platform Jaccard ---
    if hasattr(X, "toarray"):
        detected = (np.asarray((adata_raw.X > 0).sum(axis=0)).flatten() > 0)
    else:
        detected = (adata_raw.X > 0).sum(axis=0) > 0
    detected_genes = set(adata_raw.var_names[detected])

    del adata, adata_de, adata_geo
    gc.collect()
    return result, detected_genes

In [ ]:
# 4. DRIVER

In [ ]:
def main():
    rows: list[dict]   = []
    gene_sets: dict[str, set] = {}

    for folder, (block, platform, is_ref) in BLOCKS.items():
        sub_path = P9_BASE_DIR / folder / "original"
        if not (sub_path / "counts.mtx").exists():
            print(f"  [skip] missing: {sub_path}", file=sys.stderr)
            continue

        print(f"Profiling: {folder}  [{block} / {platform}{' / REF' if is_ref else ''}]")
        try:
            adata = load_real_dataset(sub_path)
            rec, detected = profile_dataset(adata, folder)
        except Exception as e:
            print(f"  [!] failed: {e}", file=sys.stderr)
            continue
        finally:
            try: del adata
            except NameError: pass
            gc.collect()

        rec.update({
            "Block": block,
            "Platform": platform,
            "Is_Reference_Platform": is_ref,
        })
        rows.append(rec)
        gene_sets[folder] = detected

    if not rows:
        print("No datasets profiled; aborting.", file=sys.stderr); sys.exit(1)

    df = pd.DataFrame(rows)

    # Column order: block / platform / metric panel / extras
    front = ["Sample", "Block", "Platform", "Is_Reference_Platform"]
    rest  = [c for c in df.columns if c not in front]
    df    = df[front + rest]

    profiling_path = OUT_DIR / "Profiling_P9.csv"
    df.to_csv(profiling_path, index=False)
    print(f"\nWrote: {profiling_path}   shape={df.shape}")

    # --- Platform-shift Δ-table (Δ vs block reference) -----------------
    shift_rows = []
    for block, sub in df.groupby("Block"):
        ref = sub[sub["Is_Reference_Platform"]]
        if ref.empty:
            print(f"  [warn] no reference platform tagged for block {block}", file=sys.stderr)
            continue
        ref_row = ref.iloc[0]
        depth_axes = ["Median_Library_Per_Cell", "Mean_Library_Per_Cell",
                      "Median_Genes_Per_Cell", "Dropout_Pct",
                      "KNN_Purity", "Mean_Pi_Score",
                      "Norm_Shannon_Entropy", "Total_CellTypes"]
        for _, row in sub.iterrows():
            entry = {"Block": block, "Sample": row["Sample"],
                     "Platform": row["Platform"],
                     "Is_Reference_Platform": row["Is_Reference_Platform"]}
            for axis in depth_axes:
                entry[axis] = row[axis]
                entry[f"Delta_{axis}"] = (
                    np.nan if row["Is_Reference_Platform"]
                    else row[axis] - ref_row[axis]
                )
                if axis == "Median_Library_Per_Cell" and row["Median_Library_Per_Cell"] > 0 and ref_row["Median_Library_Per_Cell"] > 0:
                    entry["Log10_Ratio_Median_Library_vs_Ref"] = (
                        np.nan if row["Is_Reference_Platform"]
                        else round(float(np.log10(row[axis] / ref_row[axis])), 4)
                    )
            shift_rows.append(entry)

    shift_df = pd.DataFrame(shift_rows)
    shift_path = OUT_DIR / "Phase3_PlatformShift.csv"
    shift_df.to_csv(shift_path, index=False)
    print(f"Wrote: {shift_path}   shape={shift_df.shape}")

    # --- Gene-overlap Jaccard within each block ------------------------
    block_to_folders = {}
    for folder, (block, _, _) in BLOCKS.items():
        block_to_folders.setdefault(block, []).append(folder)

    jaccard_rows = []
    for block, folders in block_to_folders.items():
        for i, fi in enumerate(folders):
            if fi not in gene_sets: continue
            for fj in folders[i:]:
                if fj not in gene_sets: continue
                si, sj = gene_sets[fi], gene_sets[fj]
                inter, union = len(si & sj), len(si | sj)
                jac = inter / union if union else np.nan
                jaccard_rows.append({
                    "Block": block,
                    "Dataset_A": fi, "Dataset_B": fj,
                    "Genes_A_detected": len(si),
                    "Genes_B_detected": len(sj),
                    "Genes_intersection": inter,
                    "Genes_union": union,
                    "Jaccard": round(jac, 4),
                })
    jacc_df = pd.DataFrame(jaccard_rows)
    jacc_path = OUT_DIR / "Phase3_GeneOverlap.csv"
    jacc_df.to_csv(jacc_path, index=False)
    print(f"Wrote: {jacc_path}   shape={jacc_df.shape}")

    # --- Plots ---------------------------------------------------------
    make_plots(df, jacc_df)
    print("\nDone.")


def make_plots(df: pd.DataFrame, jacc_df: pd.DataFrame):
    # Set style and strictly enforce Helvetica globally
    sns.set_style("white")  # standardised: white bordered panels, no gridlines (matches R theme_bench)
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    
    block_order = ["CellBench", "Pancreas", "PBMCbench"]
    metric_panel = [
        ("Median_Library_Per_Cell", "Median library per cell\n(UMI or read count)", True),
        ("Dropout_Pct",              "Dropout (% zeros)",                          False),
        ("Median_Genes_Per_Cell",    "Median genes detected per cell",            False),
        ("KNN_Purity",               "kNN purity (class separability)",           False),
        ("Mean_Pi_Score",            "Mean π (DE strength)",                      True),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    
    # Track unique legend handles and labels across panels
    legend_dict = {}

    for i, (col, label, logy) in enumerate(metric_panel):
        ax = axes[i]
        sub = df.dropna(subset=[col])
        
        sns.barplot(data=sub, x="Block", y=col, hue="Platform",
                    order=block_order, ax=ax)
        
        ax.set_title(label, fontname='Helvetica', fontweight='bold', fontsize=12)
        ax.set_xlabel("")
        ax.set_ylabel(col, fontname='Helvetica', fontsize=10)
        if logy: 
            ax.set_yscale("log")
            
        # Extract legend elements to consolidate them later, then remove from current axis
        handles, labels = ax.get_legend_handles_labels()
        for h, l in zip(handles, labels):
            if l not in legend_dict:
                legend_dict[l] = h
        ax.get_legend().remove()
        
    # Handle the 6th empty panel (bottom-right axis)
    ax_legend = axes[-1]
    ax_legend.axis("off")
    
    # Place a single, unified legend inside the empty bottom-right corner space
    if legend_dict:
        ax_legend.legend(
            handles=list(legend_dict.values()), 
            labels=list(legend_dict.keys()), 
            loc="center", 
            title="Platform",
            title_fontsize=11,
            fontsize=10,
            frameon=True
        )

    plt.tight_layout()
    out = PLOT_DIR / "phase3_platform_shift_overview.png"
    plt.savefig(out, dpi=1200, bbox_inches="tight")
    plt.close()
    print(f"  plot: {out}")

    # Gene-overlap heatmap per block
    for block, sub in jacc_df.groupby("Block"):
        names = sorted(set(sub["Dataset_A"]).union(sub["Dataset_B"]))
        mat = pd.DataFrame(index=names, columns=names, dtype=float)
        for _, r in sub.iterrows():
            mat.loc[r["Dataset_A"], r["Dataset_B"]] = r["Jaccard"]
            mat.loc[r["Dataset_B"], r["Dataset_A"]] = r["Jaccard"]
            
        fig, ax = plt.subplots(figsize=(max(5, 0.7 * len(names)), max(4, 0.7 * len(names))))
        
        # Heatmap plotting
        sns.heatmap(mat.astype(float), annot=True, fmt=".2f",
                    cmap="viridis", vmin=0, vmax=1, ax=ax,
                    cbar_kws={"label": "Jaccard of detected genes"})
        
        # Explicit font applications for heatmap text structures
        ax.set_title(f"Cross-platform gene overlap — {block}", fontname='Helvetica', fontweight='bold', fontsize=12)
        ax.set_xticklabels(ax.get_xticklabels(), fontname='Helvetica', rotation=45, ha='right')
        ax.set_yticklabels(ax.get_yticklabels(), fontname='Helvetica', rotation=0)
        
        # Update colorbar font properties dynamically
        cbar = ax.collections[0].colorbar
        cbar.ax.yaxis.label.set_fontname('Helvetica')
        for label in cbar.ax.get_yticklabels():
            label.set_fontname('Helvetica')
            
        plt.tight_layout()
        out = PLOT_DIR / f"phase3_gene_overlap_{block.lower()}.png"
        plt.savefig(out, dpi=1200, bbox_inches="tight")
        plt.close()
        print(f"  plot: {out}")


if __name__ == "__main__":
    main()